## Import Datas

In [ ]:
import sys
import os
from google.colab import drive
drive.mount('/content/drive')
folder = "/content/drive/MyDrive/Thèse de doctorat/Redaction - rapports et articles/Articles de conférence redigés/ICATH 2026/code/data/preprocessed options datas/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
import numpy as np
import random
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
TechnologyUnderlying = [ "AAPL", "AMZN", "GOOGL", "META", "MSFT" ]
FinanceUnderlying = [ "BAC", "C", "GS", "JPM", "WFC" ]
HealthUnderlying = [ "ABBV", "JNJ", "MRK", "PFE", "UNH" ]
AutomobileUnderlying = [ "F", "GM", "LCID", "RIVN", "TSLA" ]

listTickers = AutomobileUnderlying + FinanceUnderlying + HealthUnderlying + TechnologyUnderlying

In [ ]:
dataset = {
    "AAPL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "AMZN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GOOGL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "META" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "MSFT" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "BAC" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "C" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GS" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "JPM" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "WFC" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "ABBV" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "JNJ" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "MRK" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "PFE" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "UNH" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "F" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GM" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "LCID" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "RIVN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "TSLA" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
}

In [ ]:
for ticker in dataset.keys():
    train_path = os.path.join(folder, f"{ticker}_train.csv")
    test_path = os.path.join(folder, f"{ticker}_test.csv")

    if not os.path.exists(train_path):
        print(f"Missing: {train_path}")
        continue

    train_dataset = pd.read_csv(train_path)
    test_dataset = pd.read_csv(test_path)

    train_dataset = train_dataset.fillna(train_dataset.mean(numeric_only=True))
    test_dataset = test_dataset.fillna(test_dataset.mean(numeric_only=True))

    dataset[ticker]["train"] = train_dataset
    dataset[ticker]["test"] = test_dataset

## Define functions

In [ ]:
import pandas as pd
from scipy.stats import norm
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
from typing import Literal

In [ ]:
def _payoff(S: np.ndarray, K: float, option_type: Literal["call", "put"]) -> np.ndarray:
    """
    Compute the payoff of the option.

    Parameters
    ----------
    S : np.ndarray
        Underlying asset prices
    K : float
        Strike price
    option_type : str
        'call' or 'put'
    """
    if option_type == "call":
        return np.maximum(S - K, 0.0)
    elif option_type == "put":
        return np.maximum(K - S, 0.0)
    else:
        raise ValueError("option_type must be 'call' or 'put'")


def _validate_inputs(S, K, T, r, sigma, N):
    """
    Validate input parameters.
    """
    if S <= 0 or K <= 0:
        raise ValueError("S and K must be positive")
    if T <= 0 or sigma <= 0:
        raise ValueError("T and sigma must be positive")
    if N <= 0:
        raise ValueError("N must be a positive integer")


def american_binomial_price(
    S: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    N: int = 50,
    option_type: Literal["call", "put"] = "call",
) -> float:
    """
    Price an American option using the Cox-Ross-Rubinstein (CRR) binomial model.

    Parameters
    ----------
    S : float
        Current underlying asset price
    K : float
        Strike price
    T : float
        Time to maturity (in years)
    r : float
        Risk-free interest rate (annualized)
    sigma : float
        Volatility (annualized)
    N : int
        Number of time steps
    option_type : str
        'call' or 'put'

    Returns
    -------
    float
        Option price
    """
    _validate_inputs(S, K, T, r, sigma, N)

    dt = T / N
    discount = np.exp(-r * dt)

    # Up and down factors (CRR model)
    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u

    # Risk-neutral probability
    p = (np.exp(r * dt) - d) / (u - d)

    # Terminal stock prices
    j = np.arange(N + 1)
    ST = S * (u ** j) * (d ** (N - j))

    # Terminal payoff
    option_values = _payoff(ST, K, option_type)

    # Backward induction with early exercise
    for i in range(N - 1, -1, -1):
        j = np.arange(i + 1)
        ST = S * (u ** j) * (d ** (i - j))

        # Continuation value (discounted expected value)
        continuation = discount * (
            p * option_values[1:i + 2] + (1 - p) * option_values[0:i + 1]
        )

        # Immediate exercise value
        exercise = _payoff(ST, K, option_type)

        # American option: take the maximum
        option_values = np.maximum(continuation, exercise)

    return float(option_values[0])

In [ ]:
def compute_binomial_price(ticker, option_type):
  data = dataset[ticker]["test"]
  test_dataset = data[data["optionType"] == option_type]

  test_dataset['BN_price'] = test_dataset.apply(
    lambda row: american_binomial_price(
        S=row['underlyingLastPrice'],
        K=row['strike'],
        T=row["timeToMaturity"],
        r=row['riskFreeRate'],
        sigma=row['impliedVolatility'],
        N=50,
        option_type=row['optionType']
    ),
    axis=1)

  rmse = np.sqrt(mean_squared_error(test_dataset['lastPrice'], test_dataset['BN_price']))
  print(f"ticker: {ticker}, option type: {option_type}, RMSE: {rmse:.3f}")

## option price using binomial model

In [ ]:
ticker = "TSLA"
option_type="put"

In [ ]:
compute_binomial_price(ticker, option_type)